# 02 — Data profiling and relationship discovery

Run notebook 01 first. This notebook profiles schemas, missing values, duplicate candidate keys, order lifecycle dates and referential coverage. It does not modify raw data.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import pandas as pd
PROJECT=Path("/content/drive/MyDrive/ecommerce-data-engineering")
RAW=PROJECT/"data"/"raw"; REPORTS=PROJECT/"reports"
REPORTS.mkdir(parents=True,exist_ok=True)
assert len(list(RAW.glob("*.csv")))==9, "Run notebook 01 first." 

In [ ]:
table_profiles=[]
for path in sorted(RAW.glob("*.csv")):
    df=pd.read_csv(path,low_memory=False)
    table_profiles.append({"table":path.stem,"rows":len(df),"columns":len(df.columns),
        "memory_mb":round(df.memory_usage(deep=True).sum()/1024**2,2)})
    print("\\n###",path.name); print(df.dtypes.to_string()); del df
table_profile=pd.DataFrame(table_profiles).sort_values("rows",ascending=False)
display(table_profile); table_profile.to_csv(REPORTS/"table_profiles.csv",index=False)

In [ ]:
KEYS={
"olist_customers_dataset.csv":["customer_id"],
"olist_orders_dataset.csv":["order_id"],
"olist_order_items_dataset.csv":["order_id","order_item_id"],
"olist_order_payments_dataset.csv":["order_id","payment_sequential"],
"olist_order_reviews_dataset.csv":["review_id"],
"olist_products_dataset.csv":["product_id"],
"olist_sellers_dataset.csv":["seller_id"],
"product_category_name_translation.csv":["product_category_name"]}
key_results=[]; null_results=[]
for filename,key_cols in KEYS.items():
    df=pd.read_csv(RAW/filename,low_memory=False)
    key_results.append({"table":filename,"key_columns":", ".join(key_cols),
        "duplicate_key_rows":int(df.duplicated(subset=key_cols).sum()),
        "null_key_rows":int(df[key_cols].isna().any(axis=1).sum())})
    for col in df.columns:
        n=int(df[col].isna().sum())
        if n: null_results.append({"table":filename,"column":col,"null_count":n,
                                   "null_pct":round(n/len(df)*100,2)})
    del df
key_report=pd.DataFrame(key_results)
null_report=pd.DataFrame(null_results).sort_values(["table","null_pct"],ascending=[True,False])
display(key_report); display(null_report.head(40))
key_report.to_csv(REPORTS/"key_quality_report.csv",index=False)
null_report.to_csv(REPORTS/"null_profile.csv",index=False)

In [ ]:
orders=pd.read_csv(RAW/"olist_orders_dataset.csv",low_memory=False)
date_cols=["order_purchase_timestamp","order_approved_at","order_delivered_carrier_date",
"order_delivered_customer_date","order_estimated_delivery_date"]
for col in date_cols: orders[col]=pd.to_datetime(orders[col],errors="coerce",utc=True)
display(orders["order_status"].value_counts(dropna=False).rename_axis("status").reset_index(name="orders"))
print("Purchase range:",orders["order_purchase_timestamp"].min(),"to",orders["order_purchase_timestamp"].max())
display(orders.groupby("order_status")["order_delivered_customer_date"]
        .apply(lambda s:s.isna().sum()).rename("missing_delivery_timestamp").reset_index())

In [ ]:
orders=pd.read_csv(RAW/"olist_orders_dataset.csv",usecols=["order_id","customer_id"])
customers=pd.read_csv(RAW/"olist_customers_dataset.csv",usecols=["customer_id"])
items=pd.read_csv(RAW/"olist_order_items_dataset.csv",usecols=["order_id","product_id","seller_id"])
products=pd.read_csv(RAW/"olist_products_dataset.csv",usecols=["product_id"])
sellers=pd.read_csv(RAW/"olist_sellers_dataset.csv",usecols=["seller_id"])
checks=[
("orders -> customers",orders.customer_id.isin(customers.customer_id).mean()),
("items -> orders",items.order_id.isin(orders.order_id).mean()),
("items -> products",items.product_id.isin(products.product_id).mean()),
("items -> sellers",items.seller_id.isin(sellers.seller_id).mean())]
relationship_report=pd.DataFrame(checks,columns=["relationship","matched_fraction"])
relationship_report["matched_pct"]=(relationship_report.matched_fraction*100).round(3)
display(relationship_report[["relationship","matched_pct"]])
relationship_report.to_csv(REPORTS/"relationship_report.csv",index=False)

Outputs: `table_profiles.csv`, `key_quality_report.csv`, `null_profile.csv` and `relationship_report.csv`. Investigate duplicate candidate keys and null patterns before Silver transformations.